<a href="https://colab.research.google.com/github/Abish-Devcraft/DSGP-GRP11-SPATIUM/blob/furniture-classifier/notebooks/01_data_understanding_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Data Collection / Loading

In [ ]:
from pathlib import Path
ZIP_PATH = "/content/drive/MyDrive/DSGP/mit_furniture_subset.zip"   # master copy on Drive
WORK_DIR = Path("/content/mit")                                    # temporary copy on Colab's fast disk

In [ ]:
# connect Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# unzip
import os
assert os.path.exists(ZIP_PATH), f"Zip not found at {ZIP_PATH}. Check the path in the Files sidebar."

!mkdir -p {WORK_DIR}
!unzip -q -o "{ZIP_PATH}" -d {WORK_DIR}

In [ ]:
# # locate the two folders
img_dir = WORK_DIR / "indoorCVPR_09" / "Images"
ann_dir = WORK_DIR / "indoorCVPR_09annotations" / "Annotations"
print("Images folder exists:", img_dir.exists())
print("Annotations folder exists:", ann_dir.exists())

In [ ]:
# erify completeness
n_img = len(list(img_dir.rglob("*.jpg")))
n_xml = len(list(ann_dir.rglob("*.xml")))
n_rooms = len([d for d in img_dir.iterdir() if d.is_dir()])

print("Room folders:", n_rooms)
print("Photos:", n_img)
print("Annotation files:", n_xml)

assert n_img > 0 and n_xml > 0, "Nothing loaded"

In [ ]:
# check photos and labels match
img_names = {p.stem for p in img_dir.rglob("*.jpg")}
xml_names = {p.stem for p in ann_dir.rglob("*.xml")}

print("Photos without a label file:", len(img_names - xml_names))
print("Label files without a photo:", len(xml_names - img_names))

# 2. Data Understanding

In [ ]:
# list the room folders and how many photos each has
for d in sorted(img_dir.iterdir())[:5]:
    print(d.name, "->", len(list(d.glob("*.jpg"))), "photos")

In [ ]:
# read one label file to see how objects are described
sample_xml = next(ann_dir.rglob("*.xml"))
print(sample_xml)
print(sample_xml.read_text()[:1500])

In [ ]:
# draw the labelled outlines on one photo to check they match
import xml.etree.ElementTree as ET
from PIL import Image, ImageDraw

photo = img_dir / sample_xml.parent.name / (sample_xml.stem + ".jpg")
im = Image.open(photo).convert("RGB")
draw = ImageDraw.Draw(im)
for obj in ET.parse(sample_xml).getroot().findall("object"):
    pts = [(float(p.findtext("x")), float(p.findtext("y"))) for p in obj.findall("polygon/pt")]
    if len(pts) >= 3:
        draw.polygon(pts, outline="red")
        draw.text(pts[0], obj.findtext("name"), fill="yellow")
im

# 3. Exploratory Data Analysis (EDA)

In [ ]:
# count photos in each room folder and plot the top 15
from collections import Counter
import matplotlib.pyplot as plt

rooms = Counter(p.parent.name for p in img_dir.rglob("*.jpg"))
top = rooms.most_common(15)

plt.figure(figsize=(8, 5))
plt.barh([r for r, _ in top][::-1], [c for _, c in top][::-1])
plt.title("Photos per room type (top 15)")
plt.xlabel("photos")
plt.show()

print("room types:", len(rooms), "| photos:", sum(rooms.values()))

In [ ]:
# read every label file and count the raw object names
import xml.etree.ElementTree as ET

names = Counter()
for xml in ann_dir.rglob("*.xml"):
    try:
        root = ET.parse(xml).getroot()
    except ET.ParseError:
        continue
    for obj in root.findall("object"):
        names[(obj.findtext("name") or "").strip().lower()] += 1

top = names.most_common(30)
plt.figure(figsize=(8, 8))
plt.barh([n for n, _ in top][::-1], [c for _, c in top][::-1])
plt.title("Top 30 object names (raw)")
plt.show()

print("distinct names:", len(names), "| total labels:", sum(names.values()))

In [ ]:
# count raw object names inside bedroom, livingroom, kitchen and bathroom only
four = ["bedroom", "livingroom", "kitchen", "bathroom"]
per_room = {r: Counter() for r in four}

for xml in ann_dir.rglob("*.xml"):
    room = xml.parent.name
    if room not in per_room:
        continue
    try:
        root = ET.parse(xml).getroot()
    except ET.ParseError:
        continue
    for obj in root.findall("object"):
        per_room[room][(obj.findtext("name") or "").strip().lower()] += 1

for r in four:
    print(f"\n{r.upper()} | photos: {rooms[r]} | labels: {sum(per_room[r].values())}")
    for n, c in per_room[r].most_common(12):
        print(f"{c:5d}  {n}")

# 4. Data Quality Assessment

In [ ]:
# show the same object appearing under several suffix variants
import re

def base(n):
    return re.sub(r" (occluded|crop|part|top)$", "", n)

for obj in ["chair", "table", "desk", "bed"]:
    variants = {n: c for n, c in names.items() if base(n) == obj}
    print(obj, "->", variants)

In [ ]:
# counts for names that should probably be one class
synonyms = {
    "painting": ["painting", "picture", "picture frame", "poster"],
    "cabinet":  ["cabinet", "cabinets", "cupboard"],
    "table":    ["table", "small table", "side table", "conference table"],
    "pillow":   ["pillow", "cushion"],
}
for k, v in synonyms.items():
    print(k, {s: names[s] for s in v})

In [ ]:
# share of each room's labels that are structure, non-furniture or other, for all 4 rooms
four = ["bedroom", "livingroom", "kitchen", "bathroom"]
per_room = {r: Counter() for r in four}

for xml in ann_dir.rglob("*.xml"):
    room = xml.parent.name
    if room not in per_room:
        continue
    try:
        root = ET.parse(xml).getroot()
    except ET.ParseError:
        continue
    for obj in root.findall("object"):
        per_room[room][(obj.findtext("name") or "").strip().lower()] += 1

STRUCT = {"wall", "floor", "ceiling", "window", "door", "column"}
NON_FURN = {"pillow", "cushion", "lamp", "curtain", "painting", "picture", "mirror", "rug",
            "plant", "vase", "books", "book", "bottle", "plate", "bowl", "pot", "towel",
            "faucet", "sink", "stove", "toilet", "washbasin", "glass", "cup", "candle", "jar"}

for r in four:
    total = sum(per_room[r].values())
    s  = sum(c for n, c in per_room[r].items() if base(n) in STRUCT)
    nf = sum(c for n, c in per_room[r].items() if base(n) in NON_FURN)
    print(f"{r:11} structure {s/total:5.0%} | non-furniture {nf/total:5.0%} | other {1-(s+nf)/total:5.0%}")

In [ ]:
# measure the shorter side of every labelled outline in the four target rooms
MIN_SIDE = 48
sizes = []
for xml in ann_dir.rglob("*.xml"):
    if xml.parent.name not in four:
        continue
    try:
        root = ET.parse(xml).getroot()
    except ET.ParseError:
        continue
    for o in root.findall("object"):
        pts = [(float(p.findtext("x")), float(p.findtext("y"))) for p in o.findall("polygon/pt")]
        if len(pts) >= 3:
            xs, ys = zip(*pts)
            sizes.append(min(max(xs) - min(xs), max(ys) - min(ys)))

plt.hist(sizes, bins=60, range=(0, 400))
plt.axvline(MIN_SIDE, color="red", ls="--")
plt.title("Shorter side of labelled boxes (px)")
plt.show()
print(f"{sum(s < MIN_SIDE for s in sizes) / len(sizes):.0%} of boxes are under {MIN_SIDE}px")

In [ ]:
# check which wanted furniture classes exist in the data at all
base_names = Counter()
for n, c in names.items():
    base_names[base(n)] += c

wanted = {
    "wardrobe": ["wardrobe", "closet"], "dresser": ["dresser", "drawers", "drawer"],
    "nightstand": ["night table", "nightstand"], "bookshelf": ["shelf", "shelves", "bookcase"],
    "coffee_table": ["coffee table"], "side_table": ["small table", "side table"],
    "ottoman": ["ottoman"], "bar_stool": ["stool"], "tv_stand": ["tv stand"],
    "console_table": ["console table"], "sideboard": ["sideboard"], "kitchen_island": ["kitchen island"],
}
for cls, srcs in wanted.items():
    print(f"{cls:15} {sum(base_names[s] for s in srcs):5}")

In [ ]:
# share of photos that belong to our four rooms
n4 = sum(rooms[r] for r in four)
print(f"{n4} of {sum(rooms.values())} photos ({n4 / sum(rooms.values()):.0%}) are in our 4 rooms")

| # | Problem | Evidence | Fix (Step) |
|---|---------|----------|------------|
| A | Same object, many spellings | chair vs chair top / occluded / crop | Normalise names (5) |
| B | Synonyms split one class | painting / picture / poster | Merge classes (5) |
| C | Many labels are not furniture | 30% non-furniture, _20% structure | Remove (5) |
| D | Many tiny boxes | 55% under 48px | Filter (5) |
| E | Most photos are other rooms | 39% in our 4 rooms | Keep 4 rooms (5) |
| F | Missing / rare classes | tv_stand = 0, sideboard = 0 ... | Add data (6) |